In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [3]:
import os


In [4]:
%pwd
os.chdir("../")
%pwd

'd:\\PROJECTS\\CHATBOT\\The-Inner-Citadel'

In [9]:
def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
        )
    documents=loader.load()
    return documents

In [10]:
extracted_data=load_pdf_files("data")

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [ ]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunk=text_split(minimal_docs)

In [13]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

In [14]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceBgeEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2'
        )  
    return embeddings

embedding=download_hugging_face_embeddings()

C:\Users\khush\AppData\Local\Temp\ipykernel_3148\1649484688.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceBgeEmbeddings(


In [ ]:
vector=embedding.embed_query("hello people!")
len(vector) #also mentioned in model card from hugging face

In [5]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone
load_dotenv()


True

In [6]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "stoic-chatbot"


In [ ]:
from pinecone import Pinecone, ServerlessSpec


existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )


index = pc.Index(index_name)


In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)


In [15]:
#load existing index

from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [16]:
retriever=docsearch.as_retriever(search_type='similarity', search_kwargs={"k":3})

In [17]:
retriever_docs=retriever.invoke("How should I live my life?")
retriever_docs

#connect to llm to enhance response

[Document(id='bbf3f973-2319-4dac-a5f3-81ed8c478d48', metadata={'source': 'data\\daily_stoic.pdf'}, page_content='we who are being asked the question. It’s our lives that are the answer.\nNo amount of travel or reading or clever sages can tell you what you\nwant to know. Instead, it is you who must find the answer in your\nactions, in living the good life—by embodying the self-evident principles\nof justice, self-control, courage, freedom, and abstaining from evil.'),
 Document(id='2c2c5745-bc23-4776-989f-e1c9fc820d9d', metadata={'source': 'data\\daily_stoic.pdf'}, page_content='Y\nOctober 25th \nTWO TASKS\n“What, then, makes a person free from hindrance and self-\ndetermining? For wealth doesn’t, neither does high-office, state\nor kingdom—rather, something else must be found . . . in the\ncase of living, it is the knowledge of how to live.”\n—EPICTETUS, DISCOURSES, 4.1.62–64\nou have two essential tasks in life: to be a good person and to\npursue the occupation that you love. Everythi

In [18]:
from langchain_community.chat_models import ChatOllama

# Create a local Ollama chat model instance
chatModel = ChatOllama(model="phi3:mini")

# Ask a question
response = chatModel.invoke("What is the root cause of suffering?")
print(response.content)


C:\Users\khush\AppData\Local\Temp\ipykernel_3148\2501204252.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  chatModel = ChatOllama(model="phi3:mini")


The question "What is the root cause of suffering?" has been a central concern in many philosophical, religious, and psychological discussions. Various perspectives offer different answers:

1. **Philosophy** - Ancient Greek philosopher Epicurus believed that pain and suffering were primarily caused by fears regarding the afterlife (death anxiety). He suggested eliminating these unnecessary concerns to achieve a state free from suffering, known as ataraxia. Socrates also argued in Plato's "Phaedo" that knowledge of death could relieve one’ extrinsic their soul and cease experiencing pain or sorrow after physical demise (cessationism).

2. **Religions** - In Buddhism, the root cause is identified as desire—specifically ignorance leading to craving which binds beings in Samsara—the cycle of death and rebirth. By eliminating greed, hatred or ill will through ethical living (Sila), meditation (Samadhi) & developing compassionate understanding for others suffering(Karuna) practitioners can 

In [19]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

system_prompt = (
    "You are a stoic minded chatbot for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise and stoic.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


In [20]:
question_answer_chain=create_stuff_documents_chain(chatModel,prompt)
rag_chain=create_retrieval_chain(retriever, question_answer_chain)

In [21]:
response=rag_chain.invoke({"input": "what is the root cause of suffering?"})
print(response['answer'])

The root causes of suffering include fears, desires beyond need and reason, lust, envy, spite, greed, petulance, over-indulgence, attachment to impermanent things or states (such as life itself), ignorance about the nature of reality, desire for power or dominance by others, and not recognizing that true happiness lies within oneself. To overcome suffering, one must recognize these causes, understand they are relative in their importance, care only for what is right and wrong, maintain humility and simplicity, and seek God's help as the ultimate guide to freedom from such sufferings.
